# PPO DOJO â€” kit1 retrain (GPU)

1. **Runtime â†’ Change runtime type â†’ T4 GPU** first (else this trains on CPU).
2. Edit `REPO` below to your repo URL, then **Runtime â†’ Run all**.
3. Training runs in the background; the dashboard cell opens a live tab.

In [ ]:
REPO = "https://github.com/exstronium-star/pvp-bot-colab"
ITERS = 1000
ENVS = 32
TICKS = 256
OUTDIR = "runs/kit1"
RESET_POOL = True  # drop coward league snapshots from the checkpoint

In [ ]:
!test -d dojo && (cd dojo && git pull --rebase) || git clone $REPO dojo
!pip install -q numpy  # torch ships with CUDA on colab; do NOT pip install torch

In [ ]:
!nvidia-smi -L || echo 'NO GPU VISIBLE - set Runtime -> Change runtime type -> T4 GPU'
!cd dojo && python -c "import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
!cd dojo && python -m mc_pvp.validate 2>&1 | tail -2

In [ ]:
# training runs in the BACKGROUND so later cells (dashboard) can run.
# logs go to dojo/train_colab.log; checkpoints still save every 25 iters.
cmd = (f"cd dojo && nohup python -u train.py --iters {ITERS} --envs {ENVS} --ticks {TICKS} "
       f"--outdir {OUTDIR} --resume" + (" --reset-pool" if RESET_POOL else "") +
       f" > train_colab.log 2>&1 & echo started pid $!")
print("+", cmd)
!{cmd}

In [ ]:
# peek at training progress anytime (re-run this cell)
!tail -n 8 dojo/train_colab.log

In [ ]:
# live dashboard: opens in a new tab (iframe embeds get blocked by cookies).
# if the tab is empty, use the fallback monitor cell below instead.
!cd dojo && (pgrep -f "http.server 8000" >/dev/null || nohup python -m http.server 8000 >/dev/null 2>&1 &) 
!sleep 1; curl -s -o /dev/null -w "local server: %{http_code} (want 200)
" http://localhost:8000/tools/dashboard.html
from google.colab import output
output.serve_kernel_port_as_window(8000, path='/tools/dashboard.html')

In [ ]:
# fallback monitor: no server, no iframe - just re-run this cell anytime
import json, pathlib
mp = pathlib.Path('dojo/runs/kit1/metrics.jsonl')
rows = [json.loads(l) for l in mp.read_text().splitlines() if l.strip()] if mp.exists() else []
pts = [r for r in rows if 'ret' in r]
evs = [r for r in rows if r.get('event') == 'eval']
print(f'{len(rows)} points | {mp} exists={mp.exists()}')
if pts:
    t = pts[-1]
    print(f"iter {t['iter']} ret {t.get('ret')} ent {t.get('ent')} eps {t.get('eps')}")
if evs:
    e = evs[-1]
    print(f"eval it {e['iter']}: win {e['win_rate']*100:.1f}% loss {e['loss_rate']*100:.1f}% dmg {e['dmg_dealt']:.1f}/{e['dmg_taken']:.1f}")
if pts:
    import matplotlib.pyplot as plt
    xs = [r['iter'] for r in pts]
    plt.figure(); plt.plot(xs, [r.get('ret') for r in pts]); plt.title('avg reward'); plt.xlabel('iter'); plt.show()
    if evs:
        plt.figure(); plt.plot([r['iter'] for r in evs], [r['win_rate']*100 for r in evs]); plt.title('eval win %'); plt.xlabel('iter'); plt.show()
else:
    print('no data yet - training writes its first point after ~10 iters; check the tail cell')

In [ ]:
# when training is done: pull the weights back to your PC
from google.colab import files
files.download('dojo/runs/kit1/ppo.pt')